In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
# !source drive/MyDrive/AI-Framework/venv/bin/activate

In [3]:
# import sys
# import os
# sys.path.insert(0, '/content/drive/MyDrive/AI-Framework')
# os.chdir('/content/drive/MyDrive/AI-Framework')

In [4]:
import mediapipe as mp
import torch

from pathlib import Path
from matplotlib import pyplot as plt

from Network.Network import NYOLO, NMediaPipe
from Dataset.Dataset import COCO, VOC
from Train.Train import TYOLO
from Validation.DetectionSegmentation import ValDetectionSegmentation
from Validation.PoseEstimation import ValPoseEstimation
from Network.YoloIPC8 import IPC8Predictor

In [5]:
dataset = COCO(
    image_dirs={
            "train": "/home/orchid/ipc_yolo/dataset/coco2017/train2017",
            "valid": "/home/orchid/ipc_yolo/dataset/coco2017/val2017",
            "test": "/home/orchid/ipc_yolo/dataset/coco2017/test2017",
    },
    annotation_dirs={
            "train": "/home/orchid/ipc_yolo/dataset/coco2017/annotations",
            "valid": "/home/orchid/ipc_yolo/dataset/coco2017/annotations",
            "test": "/home/orchid/ipc_yolo/dataset/coco2017/annotations",
    },
    class_file="dataset/custom_coco_yolo/object.names",
    data_file="dataset/custom_coco_yolo/obj.data",
    batch_size=4,
    split="train",
    out_dir="dataset/custom_coco_yolo",
    with_ground_truth=True,
    resize_mode="letterbox",
    image_size=608,
    mosaic_enable=False,
    multiscale=True,
    multiscale_min_size=320,
    multiscale_max_size=512,
    multiscale_step=32,
    multiscale_interval=64)

# dataset = VOC(
#     image_dirs={
#             "train": "dataset/VOC/VOC2012_train_val/ImageSets",
#             "valid": "dataset/VOC/VOC2012_train_val/ImageSets",
#             "test": "dataset/VOC/VOC2012_test/ImageSets",
#     },
#     annotation_dirs={
#             "train": "dataset/VOC/VOC2012_train_val/Annotations",
#             "valid": "dataset/VOC/VOC2012_train_val/Annotations",
#             "test": "dataset/VOC/VOC2012_test/Annotations",
#     },
#     class_file="dataset/VOC/object.names",
#     data_file="dataset/VOC/obj.data",
#     batch_size=16,
#     split="train",
#     out_dir="dataset/VOC",
#     with_ground_truth=True,
#     resize_mode="letterbox",
#     image_size=608,
#     multiscale=True,
#     multiscale_min_size=320,
#     multiscale_max_size=512,
#     multiscale_step=32,
#     multiscale_interval=64)

dataset.get_stats_for_split('dataset/custom_coco_yolo', 'valid')


--- STATISTICS FOR: VAL ---
Total Images: 2948
Class 0 (person    ):  11004 instances (78.39%)
Class 1 (car       ):   1932 instances (13.76%)
Class 2 (motorbike ):    371 instances (2.64%)
Class 3 (truck     ):    415 instances (2.96%)
Class 4 (bicycle   ):    316 instances (2.25%)


In [ ]:
# yolo detection and segmentation algorithm

yolo = NYOLO("ipc8", "n", num_classes=dataset.class_num, task='instance_segmentation', use_sahi=False)
yolo.build()
# train = TYOLO(yolo, dataset, learning_rate=1e-3, 
#                 cost_function_name='yolo_cost_function', 
#                 isScheduled=True, 
#                 scheduler_name="OneCycleLR", 
#                 max_lr=0.001, 
#                 total_steps=20,
#                 pct_start=0.1,
#                 final_div_factor=100, 
#             )
# train.start(f'{args.model_backup}/checkpoint', epoch_num=args.epoch_num, report_epoch=1)
yolo.model.load_state_dict(torch.load(f'dataset/global/backups/checkpoint_nano_last.pt')['model_state_dict'])
yolo.export_onnx(
    f"dataset/global/backups/checkpoint_nano_last.onnx",
    f"dataset/global/backups/checkpoint_nano_last.pt",
    input_size=(768, 768),
    metadata_json_path=f"dataset/global/backups/out.json",
    max_input_side=768,
)
yolo.build_predictor(dataset.target_classes, max_input_size=768)
val = ValDetectionSegmentation(yolo.predictor, dataset, conf_tresh=0.5, iou_tresh=.5, task='instance_segmentation')
# val.test_performance(output_dir="dataset/global/detection_outputs", source="/home/orchid/PythonProject/dataset/custom_coco_yolo/videos/saved_frames")

In [ ]:
# from Validation.FallDetection import ValFallDetection

# # val.test_from_video(Path(f'dataset/custom_coco_yolo/videos/cam5.avi'), 
# #                      labels_dir= Path('dataset/custom_coco_yolo/videos'),
# #                      visuals_dir=Path(f'dataset/custom_coco_yolo/videos'),
# #                      save_visuals=True, save_txt=False)

# fall_val = ValFallDetection(yolo.predictor, dataset, conf_tresh=0.5)
# fall_val.test_from_video(Path(f'dataset/custom_coco_yolo/videos/cort_all_falls.mkv'), 
#                         labels_dir= Path('dataset/custom_coco_yolo/videos'),
#                         visuals_dir=Path(f'dataset/custom_coco_yolo/videos'),
#                         save_visuals=True, save_txt=False)

In [ ]:
# from Network.Network import NMediaPipe
# mediapipe = NMediaPipe('mediapipe', 
#                     task='detection',
#                     model_path='/home/orchid/PythonProject/dataset/custom_coco_yolo/backups/efficientdet_lite2 (float16).tflite', 
#                     detection_max_results=50, 
#                     detection_score_threshold=0.5,
#                     class_names=dataset.target_classes,
#                     use_gpu=False,
#                     use_sahi=True,
#                     )
# mediapipe.build()
# val = ValDetectionSegmentation(mediapipe.build_predictor(max_input_size=768), dataset, conf_tresh=0.5, iou_tresh=.5, task='detection')
# val.test_performance(output_dir="dataset/global/detection_outputs", source="/home/orchid/PythonProject/dataset/custom_coco_yolo/videos/saved_frames")

In [ ]:
# # mediapipe pose_estimation algorithm
# media_pipe = NMediaPipe(name='mediapipe', 
#                         task='pose_estimation', 
#                         model_path="/home/orchid/PythonProject/dataset/custom_coco_yolo/backups/pose_landmarker_heavy.task",
#                         output_segmentation_masks=False,
#                         num_poses=10,
#                         min_pose_detection_confidence=0.5,
#                         min_tracking_confidence=0.5,
#                         use_gpu=True
#                         )
# media_pipe.build()
# media_pipe.build_predictor()

# val = ValPoseEstimation('mediapipe', media_pipe.predictor, dataset)
# val.test_performance('dataset/custom_coco_yolo/pose_estimation_outputs', source='/home/orchid/PythonProject/dataset/custom_coco_yolo/videos/saved_frames', save_visuals=True, save_txt=True)

In [ ]:
# val.test_from_video(Path(f'dataset/custom_coco_yolo/videos/me_walking.mkv'), 
#                      labels_dir= Path('dataset/custom_coco_yolo/videos'),
#                      visuals_dir=Path(f'dataset/custom_coco_yolo/videos'),
#                      save_visuals=True, save_txt=False)

# val.test_from_video(Path(f'dataset/custom_coco_yolo/videos/brewery_cam.mkv'), 
#                      labels_dir= Path('dataset/custom_coco_yolo/videos'),
#                      visuals_dir=Path(f'dataset/custom_coco_yolo/videos'),
#                      save_visuals=True, save_txt=False)

# val.test_from_video(Path(f'dataset/custom_coco_yolo/videos/parking_lot.mkv'), 
#                      labels_dir= Path('dataset/custom_coco_yolo/videos'),
#                      visuals_dir=Path(f'dataset/custom_coco_yolo/videos'),
#                      save_visuals=True, save_txt=False)

# val.test_from_video(Path(f'dataset/custom_coco_yolo/videos/vide01.mp4'), 
#                      labels_dir= Path('dataset/custom_coco_yolo/videos'),
#                      visuals_dir=Path(f'dataset/custom_coco_yolo/videos'),
#                      save_visuals=True, save_txt=False)

val.start()